# Tool Call Label Masking in Oumi

This notebook walks through exactly how `DataCollatorForCompletionOnlyLM` decides which tokens are **trained on** (unmasked) and which are **ignored** (label = -100) when a conversation contains tool calls — and why it gets the tool result wrong.

We then demonstrate the new `ToolAwareCompletionsCollator` that fixes this.

## 1. Setup

In [ ]:
import json

from IPython.display import HTML, display
from transformers import AutoTokenizer

from oumi.core.collators.tool_aware_completions_collator import (
    ToolAwareCompletionsCollator,
)
from oumi.core.collators.trl_data_collator_for_completion_only_lm import (
    DataCollatorForCompletionOnlyLM,
)

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer loaded:", MODEL_NAME)

## 2. Build a tool-calling conversation

A tool-calling exchange has four turns:

| Turn | Role | Content |
|------|------|---------|
| 1 | `user` | Asks a question |
| 2 | `assistant` | Issues a **tool call** — no text answer yet |
| 3 | `tool` | Returns the **tool result** — produced by your code |
| 4 | `assistant` | Gives the **final answer** using the result |

In [ ]:
tool_call_content = json.dumps(
    {"name": "get_weather", "arguments": {"location": "Paris"}}
)

messages = [
    {"role": "user", "content": "What's the weather in Paris?"},
    # Turn 2 — TOOL CALL: assistant decides to call a function
    {"role": "assistant", "content": f"<tool_call>\n{tool_call_content}\n</tool_call>"},
    # Turn 3 — TOOL RESULT: your code ran get_weather() and returned this
    {"role": "tool", "content": json.dumps({"result": "Sunny, 18°C"})},
    # Turn 4 — FINAL ANSWER: assistant reads the result and answers the user
    {"role": "assistant", "content": "The weather in Paris is sunny and 18°C."},
]

rendered = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=False
)
print(rendered)

Notice the role markers SmolLM2 uses:
- User turn:      `<|im_start|>user\n`
- Tool result:    `<|im_start|>tool\n`  ← **different from user**
- Assistant turn: `<|im_start|>assistant\n`

This is why the old collator fails: it only searches for `<|im_start|>user\n` as the instruction boundary, so `<|im_start|>tool\n` is invisible to it.

## 3. Visualisation helper

In [ ]:
def show_masking(input_ids: list[int], labels: list[int], tokenizer) -> None:
    """Render color-coded token masking. Green = trains, Red = masked."""
    parts = []
    masked_style = (
        "background:#ffcccc;border:1px solid #ff9999;"
        "border-radius:3px;padding:1px 2px;margin:1px;"
    )
    unmasked_style = (
        "background:#ccffcc;border:1px solid #66cc66;"
        "border-radius:3px;padding:1px 2px;margin:1px;"
    )
    for tok_id, lbl in zip(input_ids, labels):
        text = tokenizer.decode([tok_id])
        text = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
        text = text.replace(" ", "&nbsp;").replace("\n", "↵<br/>")
        if lbl == -100:
            style = masked_style
            title = "masked (label=-100)"
        else:
            style = unmasked_style
            title = f"label={lbl}"
        parts.append(f'<span style="{style}" title="{title}">{text}</span>')
    display(
        HTML(
            '<p style="font-family:monospace;line-height:2;">' + "".join(parts) + "</p>"
        )
    )


def summarise(name: str, labels: list[int], total: int) -> None:
    """Print a one-line summary of how many tokens are unmasked."""
    unmasked = sum(1 for v in labels if v != -100)
    print(f"{name}: {unmasked}/{total} tokens unmasked ({100 * unmasked / total:.1f}%)")


token_ids = tokenizer.encode(rendered, add_special_tokens=False)
N = len(token_ids)
print(f"Total tokens: {N}")

---
## Old collator: Case A — Without `instruction_template`

Finds the **last** `response_template` and masks everything before it.
Only the final assistant turn trains.

```python
# trl_data_collator_for_completion_only_lm.py line 122
batch["labels"][i, :response_token_ids_end_idx] = self.ignore_index
```

In [ ]:
RESPONSE_TEMPLATE = "<|im_start|>assistant\n"

old_no_inst = DataCollatorForCompletionOnlyLM(
    response_template=RESPONSE_TEMPLATE,
    instruction_template=None,
    tokenizer=tokenizer,
)
b = old_no_inst.torch_call([token_ids])
labels_A = b["labels"][0].tolist()
summarise("Case A", labels_A, N)
show_masking(b["input_ids"][0].tolist(), labels_A, tokenizer)

**Expected:** Only the final assistant answer is green. Tool call turn is masked.

---
## Old collator: Case B — With `instruction_template` ← the bug

Pairs `<|im_start|>user\n` with `<|im_start|>assistant\n`.  
Because `<|im_start|>tool\n` is not recognised, the tool result leaks out unmasked.

```python
# pairs: zip([pos_user], [pos_asst_1, pos_asst_2])
# zip stops at len=1 → only masks labels[:pos_asst_1]
# everything after the first assistant marker is UNMASKED
```

In [ ]:
INSTRUCTION_TEMPLATE = "<|im_start|>user\n"

old_with_inst = DataCollatorForCompletionOnlyLM(
    response_template=RESPONSE_TEMPLATE,
    instruction_template=INSTRUCTION_TEMPLATE,
    tokenizer=tokenizer,
)
b2 = old_with_inst.torch_call([token_ids])
labels_B = b2["labels"][0].tolist()
summarise("Case B", labels_B, N)
show_masking(b2["input_ids"][0].tolist(), labels_B, tokenizer)

**Bug visible:** The `<|im_start|>tool\n...` block (tool result) is green — it is being trained on, which is incorrect.

---
## New collator: Case C — Train on all assistant turns (including tool calls)

`ToolAwareCompletionsCollator` starts with **all labels = -100**, then unmasks
only the spans between each `response_template` and the next `end_of_turn_template`.
It never looks at user/tool markers at all.

```python
# core algorithm (tool_aware_completions_collator.py)
batch["labels"][i, :] = ignore_index                    # start: all masked
for each response_template position:
    find next end_of_turn_template → content_end
    batch["labels"][i, content_start:content_end] = input_ids  # unmask span
```

In [ ]:
END_OF_TURN = "<|im_end|>"

new_all_asst = ToolAwareCompletionsCollator(
    response_template=RESPONSE_TEMPLATE,
    end_of_turn_template=END_OF_TURN,
    mask_tool_calls=False,  # train on tool calls too
    tokenizer=tokenizer,
)
b3 = new_all_asst.torch_call([token_ids])
labels_C = b3["labels"][0].tolist()
summarise("Case C", labels_C, N)
show_masking(b3["input_ids"][0].tolist(), labels_C, tokenizer)

**Expected:**
- User turn → red
- Assistant tool_call turn → **green** (model learns to generate tool calls)
- Tool result (`<|im_start|>tool\n...`) → **red** (correctly masked)
- Assistant final answer → **green**

---
## New collator: Case D — Train on final answers only (mask tool calls too)

Pass `mask_tool_calls=True` and `tool_call_start_template="<tool_call>"`.  
Any assistant span containing `<tool_call>` is re-masked, so only plain-text
final answers contribute to the loss.

In [ ]:
new_final_only = ToolAwareCompletionsCollator(
    response_template=RESPONSE_TEMPLATE,
    end_of_turn_template=END_OF_TURN,
    mask_tool_calls=True,
    tool_call_start_template="<tool_call>",
    tokenizer=tokenizer,
)
b4 = new_final_only.torch_call([token_ids])
labels_D = b4["labels"][0].tolist()
summarise("Case D", labels_D, N)
show_masking(b4["input_ids"][0].tolist(), labels_D, tokenizer)

**Expected:**
- User turn → red
- Assistant tool_call turn → **red** (masked because it contains `<tool_call>`)
- Tool result → **red**
- Assistant final answer → **green**

---
## Summary

| Turn | Role | Case A | Case B (bug) | Case C (new) | Case D (new) |
|------|------|--------|--------------|--------------|---------------|
| 1 | user | masked | masked | masked | masked |
| 2 | assistant (tool_call) | masked | unmasked | **unmasked** | masked |
| 3 | tool (tool result) | masked | **unmasked ← bug** | masked | masked |
| 4 | assistant (final answer) | unmasked | unmasked | **unmasked** | **unmasked** |

**Which to use:**
- **Case C** (`mask_tool_calls=False`): use when you want the model to learn to *generate* tool calls as well as final answers.
- **Case D** (`mask_tool_calls=True`): use when tool call arguments are fixed/known and you only care about training the final response quality.

## Raw label tables

In [ ]:
def print_label_table(name, input_ids, labels, tokenizer, max_rows=80):
    """Print a token-by-token table showing IDs and masking status."""
    print(f"=== {name} ===")
    print(f"{'idx':>5}  {'tok_id':>7}  {'label':>7}  text")
    print("-" * 52)
    for i, (tok, lbl) in enumerate(zip(input_ids, labels)):
        if i >= max_rows:
            print(f"  ... ({len(input_ids) - max_rows} more rows)")
            break
        text = repr(tokenizer.decode([tok]))
        flag = "MASKED" if lbl == -100 else ""
        print(f"{i:>5}  {tok:>7}  {lbl:>7}  {text}  {flag}")
    print()


ids = b3["input_ids"][0].tolist()  # same input_ids for all cases
print_label_table(
    "Case C — new collator, train all assistant", ids, labels_C, tokenizer
)
print_label_table("Case D — new collator, final answers only", ids, labels_D, tokenizer)